# Domain decomposition in ExaHyPE 2

As ExaHyPE is built on top of Peano, its prime parallelisation strategy is domain decomposition with non-overlapping subdomains split along the Peano space-filling curve (SFC). That is, all the cells are enumerated along the Peano SFC, then cut into chunks and each chunk forms a subdomain. If you have MPI enabled, each chunk goes onto an MPI rank. If you have multiple threads, then we cut each chunk again and the resulting subchunks then are handled by different threads.

So the domain decomposition is applied to both MPI and multithreading. Without MPI, that's one way to use the threads. Using the domain decomposition without any subchunks for the threads is also possible. In this case, you might want to use tasking, which is a different tutorial. 



## Prepare the setup

We assume that you have configured your code with MPI and/or multithreading by adding

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
--with-mpi=myMPICall --with-multithreading=Pick
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Invoke the automake script with `--help` to see what options are on the table. In CMake, all configuration works analogously. After that, you have to rebuild your whole code. Once complete, all **ExaHyPE 2** applications will pick up these settings for their code, i.e. we can continue to execute this notebook.


## Configuring the environment

The first thing we will do is importing the **Peano 4** and **ExaHyPE 2** API. While we could do this in the terminal via `export PYTHONPATH=$PYTHONPATH:$PEANO_ROOT/python`, we again work directly within the notebook. Once on the Python path, the core packages can be imported just like any other package:

In [ ]:
import sys
!{sys.executable} -m pip install --break-system-packages scipy mpmath sympy

If some of these imports fail due to missing packages, either install through your preferred system environment or install them straightaway from within the notebook (this is not the elegant version, as you might spot/derive from the `--break-system-packages` attribute):

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../../../python/"))

import peano4.api, exahype2

## Demo solver

To make the tutorial do something, we pick one of our simplest exercise solvers, create an **ExaHyPE 2** project, and add this solver to the project:

In [ ]:
exahype2_project = exahype2.Project(
    namespace=["tutorials", "exahype2", "exercises", "acoustic"],
    project_name="Acoustic",
    directory=".",
    executable="exahype2-acoustic"
)

In [ ]:
dimensions = 2
patch_size = 16
cell_h = 1.1 * 2.0 / (3.0 ** 3)

my_solver = exahype2.solvers.fv.rusanov.GlobalFixedTimeStep(
    name                  = "PlanarAcousticSolver",
    patch_size            = patch_size,
    min_volume_h          = cell_h/patch_size,
    max_volume_h          = cell_h/patch_size,
    normalised_time_step_size = 0.01,
    unknowns              = 3,
    auxiliary_variables   = 0
)

All the implementation is ready to go in the tutorials section. Nothing here is special - we simply follow the (solutions of the) exercises.

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation
)

### The ExaHyPE 2 project:

The solver setup is not in any way surprising either: 

In [ ]:
!mkdir solutions

In [ ]:
exahype2_project.add_solver(my_solver)
exahype2_project.set_output_path("solutions")

In [ ]:
exahype2_project.set_global_simulation_parameters(
    dimensions            = 2,
    size                  = [2.0,  2.0],
    offset                = [-1.0, -1.0],
    min_end_time          = 1.414,  # sqrt(2)
    max_end_time          = 1.414,  # sqrt(2)
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.1,
    periodic_BC           = [True, True],
)

### Activate the domain decomposition

The domain decomposition is introduced by adding one line to the script:

In [ ]:
exahype2_project.set_load_balancing( "toolbox::loadbalancing::strategies::RecursiveBipartition",
                                    "new ::exahype2::LoadBalancingConfiguration()" )

### Standard workflow

For context on available strategies and diagnostics, see the toolbox overview in @ref toolbox_loadbalancing.

Now we kick the lowering off:

In [ ]:
exahype2_project.set_Peano4_installation("../../../", peano4.api.output.CompileMode.Release)
peano4_project = exahype2_project.generate_Peano4_project()
peano4_project.generate()

and follow the standard approach to add physics, run the benchmark and visualise the outcome. We leave this up to the user to add this here to the tutorial:

In [ ]:
# to be implemented by user

# Tailoring the load balancing

Before we start a conversation about various load balancing flavours, it makes sense to study an outcome of the run above. First of all, the run will yield a couple of subpartitions, as we have now set a load balancing flavour. Next, it makes sense to open any output in your visualisation tool of choice and to recognise that the plotted data now has an additional field. If you pick it, you can visualise the arising domain decomposition. 

Once we have confirmed these properties, it might also be the right point to study ExaHyPE's and Peano's pages on performance assessment and optimisation. There are several other tools (scripts) that can help us to understand and digest what the load balancing looks like. In a lot of cases, we will next want to tune our load balancing:

## Picking a load balancing strategy

The `toolbox::loadbalancing::strategies::RecursiveBipartition` from above is only one load balancing strategy out of many that we have on offer with ExaHyPE. Have a look at the documentation of the load balancing toolbox for further information. 

In [ ]:
exahype2_project.set_load_balancing( "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
                                    "new ::exahype2::LoadBalancingConfiguration()" )

## Tailoring the load balancing strategy

Once you have committed to a strategy, you can amend its configuration by tweaking the configuration object. Study the constructor of `exahype2::LoadBalancingConfiguration` and alter the arguments therein to pick minimal tree sizes, the maximum number of subdomains per rank, and so forth:

In [ ]:
exahype2_project.set_load_balancing( "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
                                    "new ::exahype2::LoadBalancingConfiguration(alter the arguments here)" )

### Example: real configuration (min tree size, max trees, mode)

In [ ]:
# Balanced, bottom-up splitting; ensure trees aren’t too small and allow up to 2x threads after initial distribution
exahype2_project.set_load_balancing(
    "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
    "new ::exahype2::LoadBalancingConfiguration("
    "0.95, /* loadBalancingQuality */ "
    "512,  /* minSizeOfTree (cells) */ "
    "false,/* assumePeriodicBoundaryConditions */ "
    "::exahype2::LoadBalancingConfiguration::UseNumberOfThreads,       /* max trees during initial distribution */ "
    "::exahype2::LoadBalancingConfiguration::UseTwiceTheNumberOfThreads,/* max trees afterwards */ "
    "peano4::SplitInstruction::Mode::BottomUp                          /* split mode */ "
    ")"
)

In [ ]:
# More aggressive top-down splitting; smaller trees and a fixed small cap on trees
exahype2_project.set_load_balancing(
    "toolbox::loadbalancing::strategies::SpreadOutOnceGridStagnates",
    "new ::exahype2::LoadBalancingConfiguration("
    "0.98, /* loadBalancingQuality */ "
    "256,  /* minSizeOfTree (cells) */ "
    "true, /* assumePeriodicBoundaryConditions */ "
    "2,    /* max trees during initial distribution */ "
    "4,    /* max trees afterwards */ "
    "peano4::SplitInstruction::Mode::AggressiveTopDown /* split mode */ "
    ")"
)

For additional tuning guidance, see the strategy overview in @ref page_toolbox_loadbalancing_strategies and the offline API in @ref page_toolbox_loadbalancing_api.


# Offline load balancing

Offline load balancing is the crème de la crème for a lot of ExaHyPE applications. Basically, you run your simulation of choice once but you stop it immediately after the mesh is constructed and a first checkpoint has been written. After that, you take the resulting patch files and you hand them over to a Python script which computes the optimal domain decomposition. A posteriori, this is possible.

## Feed patch files into offline algorithm

In [ ]:
!python3 ../../../src/toolbox/loadbalancing/api/OfflineDecomposition.py -f Input/example -d 3 -p -s 2

## Feed offline outcome into load balancing

To feed the outcome of the offline calculation into the solver again, we employ the loadbalancer `toolbox::loadbalancing::strategies::Hardcoded`:

In [ ]:
# Example: paste the lists printed by display_hardcoded_split_parameters(...)
exahype2_project.set_load_balancing( "toolbox::loadbalancing::strategies::Hardcoded",
                                    "{ {2, 4, 6}, {0, 0, 2}, {128, 256, 128}, {1, 2, 3}, peano4::SplitInstruction::Mode::AggressiveTopDown }" )

Interpreting the lists (offline extraction):
The Python helper [`extract_hardcoded_split_parameters`](src/toolbox/loadbalancing/api/strategies/SplitTrees.py) returns
`(gridSweeps, splittingTrees, numberOfCells, subTrees)`. Here:
- `gridSweeps`: `base_sweep * (level + 1)` (levels are 0-based in the API tree arrays). The helper can emit multiple
  entries per subtree, one per level where it is introduced.
- `splittingTrees`: inferred unique parent subtree id for the introduced cells on that level (or 0 if ambiguous).
- `numberOfCells`: count of introduced cells on that level, i.e. cells with the target subtree id whose parent exists
  (level>0) and has a different subtree id. For level 0, it is the number of cells on level 0 that already belong to
  the target subtree.
- `subTrees`: subtree ids introduced by the splits (not ranks). You must map these subtree ids to destination MPI ranks for the Hardcoded constructor’s destinationRanks list.
If you paste the raw output, ensure that `{subTrees}` is converted into actual rank ids in your setup.

C++ equivalent: instantiate the Hardcoded strategy directly
```cpp
#include "toolbox/loadbalancing/strategies/Hardcoded.h"

auto* lb = new toolbox::loadbalancing::strategies::Hardcoded(
  /* gridSweeps      */ {2, 4, 6},
  /* splittingTrees  */ {0, 0, 2},
  /* numberOfCells   */ {128, 256, 128},
  /* destinationRanks*/ {1, 2, 3},
  /* modes           */ {peano4::SplitInstruction::Mode::AggressiveTopDown,
                         peano4::SplitInstruction::Mode::AggressiveTopDown,
                         peano4::SplitInstruction::Mode::AggressiveTopDown}
);

// Or, if all splits use the same mode:
auto* lb2 = new toolbox::loadbalancing::strategies::Hardcoded(
  {2, 4, 6}, {0, 0, 2}, {128, 256, 128}, {1, 2, 3},
  peano4::SplitInstruction::Mode::AggressiveTopDown
);
```